In [ ]:
library(Seurat)
library(Matrix)
library(dplyr)
library(tibble)

# =========================
# paths and object settings
# =========================
seurat_rds <- "/path/to/nature_study/RNA_final_results/rna_full_final.rds"
outdir <- "/path/to/nature_study/RNA_export_for_stats"
dir.create(outdir, recursive = TRUE, showWarnings = FALSE)

celltype_col <- "celltype_rna"
donor_col <- "donor"
condition_col <- "condition"
celltype_keep <- "HSC_MPP"

progenitor_levels <- c("HSC_MPP", "B_primed_prog", "erythroid_prog", "erythroid_heme_prog")

# =========================
# load object
# =========================
rna <- readRDS(seurat_rds)

stopifnot(all(c(celltype_col, donor_col, condition_col) %in% colnames(rna@meta.data)))

meta <- rna@meta.data %>%
  tibble::rownames_to_column("cell")

table(meta[[celltype_col]], useNA = "ifany")
table(meta[[condition_col]], useNA = "ifany")
table(meta[[donor_col]], useNA = "ifany")

# =========================
# A1. export HSC_MPP donor pseudobulk counts
# =========================
meta_hsc <- meta %>%
  filter(.data[[celltype_col]] == celltype_keep) %>%
  mutate(group_id = paste(.data[[condition_col]], .data[[donor_col]], sep = "__"))

stopifnot(nrow(meta_hsc) > 0)

counts <- GetAssayData(rna, assay = DefaultAssay(rna), layer = "counts")

# keep only HSC_MPP cells
counts_hsc <- counts[, meta_hsc$cell, drop = FALSE]

# pseudobulk by donor within HSC_MPP
group_ids <- unique(meta_hsc$group_id)

pb_list <- lapply(group_ids, function(g) {
  cells_g <- meta_hsc$cell[meta_hsc$group_id == g]
  Matrix::rowSums(counts_hsc[, cells_g, drop = FALSE])
})

pb_mat <- do.call(cbind, pb_list)
colnames(pb_mat) <- group_ids
rownames(pb_mat) <- rownames(counts_hsc)

pb_meta <- meta_hsc %>%
  select(group_id, donor = all_of(donor_col), condition = all_of(condition_col)) %>%
  distinct() %>%
  arrange(condition, donor)

pb_mat <- pb_mat[, pb_meta$group_id, drop = FALSE]

print(dim(pb_mat))
print(pb_meta)
print(colSums(pb_mat))

saveRDS(pb_mat, file.path(outdir, "HSC_MPP_pseudobulk_counts.rds"))
write.table(
  pb_meta,
  file = file.path(outdir, "HSC_MPP_pseudobulk_coldata.tsv"),
  sep = "\t", quote = FALSE, row.names = FALSE
)

# optional: export panel-gene counts too
panel_genes <- c("EBF1", "PAX5", "CD24", "DNTT", "BCL11A", "SPI1", "IRF4", "IRF8")
panel_genes_present <- intersect(panel_genes, rownames(pb_mat))
pb_panel <- pb_mat[panel_genes_present, , drop = FALSE]

write.table(
  as.data.frame(pb_panel) %>% rownames_to_column("gene"),
  file = file.path(outdir, "HSC_MPP_pseudobulk_counts_panel.tsv"),
  sep = "\t", quote = FALSE, row.names = FALSE
)

# =========================
# A2. export progenitor metadata for composition models
# =========================
meta_prog <- meta %>%
  filter(.data[[celltype_col]] %in% progenitor_levels) %>%
  mutate(
    is_B_primed_prog = ifelse(.data[[celltype_col]] == "B_primed_prog", 1L, 0L)
  ) %>%
  select(
    cell,
    donor = all_of(donor_col),
    condition = all_of(condition_col),
    celltype = all_of(celltype_col),
    is_B_primed_prog
  )

stopifnot(nrow(meta_prog) > 0)

write.table(
  meta_prog,
  file = file.path(outdir, "progenitor_cell_metadata_for_composition.tsv"),
  sep = "\t", quote = FALSE, row.names = FALSE
)

# donor-level counts export too
prog_counts <- meta_prog %>%
  group_by(donor, condition) %>%
  summarise(
    n_prog = n(),
    n_B = sum(is_B_primed_prog),
    frac_B = n_B / n_prog,
    .groups = "drop"
  ) %>%
  arrange(condition, donor)

write.table(
  prog_counts,
  file = file.path(outdir, "progenitor_donor_counts_for_composition.tsv"),
  sep = "\t", quote = FALSE, row.names = FALSE
)

prog_counts